> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# Amazon Neptune 图存储

## 概览

本 notebook 讲 Semantica 的 Amazon Neptune 数据库集成。Neptune 是全托管的图数据库服务，既支持属性图（OpenCypher/Gremlin），也支持 RDF 图（SPARQL）。

### 先看主线

这是 09 篇的云端孪生篇：同样的 `GraphStore` 接口，后端从 Neo4j/FalkorDB 换成 Neptune——**代码几乎不用改，这正是多后端抽象的价值**。但云上多出三件事，本篇三段：

**上云（准备 + 第 1 步）—— 开环境**

CloudFormation 一键开栈（带成本与安全提醒）、IAM 认证连接。

**同接口 CRUD（第 2–6 步）—— 熟悉的建库用库**

节点、关系、OpenCypher、图分析、统计——09 篇"建库/用库/养库"的节奏原样走一遍。

**懂差异（第 7 步 + 特性节）—— Neptune 的脾气**

连接管理、原生 `~id`、OpenCypher 与 Neo4j 的差异、OLTP 与 OLAP 的分工。

### 核心特性

- **IAM 认证**：经 AuthManager 用 AWS SigV4 签名安全访问
- **OpenCypher 支持**：标准 OpenCypher 语法查询
- **Bolt 协议**：用 Neo4j Bolt 驱动做高效二进制通信
- **原生 ~id 支持**：利用 Neptune 原生的元素 ID 处理
- **完整 CRUD**：节点和关系的增删改查
- **自动重试**：内置指数退避重试，应对瞬时错误

### 前置条件

- 一个 Amazon Neptune 数据库集群
- 配置好 AWS 凭据（boto3、环境变量或 IAM 角色）
- 能网络访问 Neptune 集群（VPC、安全组）
- 你的公网 IP 或 VPN/办公网 CIDR（跑 `curl ifconfig.me` 查公网 IP），下面用来限制数据库访问

#### 用 CloudFormation 快速搭建

没有 Neptune 集群？用提供的 CloudFormation 模板建一个（带公网端点和 IAM 认证）：

```bash
# 部署 Neptune 栈（约 15-20 分钟）
# 把 203.0.113.25/32 换成你自己的公网 IP（curl ifconfig.me 查）
# 或办公/VPN 网段。这在网络层面限制谁能访问数据库——
# 除非短时本地实验，绝不要放宽到 0.0.0.0/0。
aws cloudformation create-stack \
  --stack-name semantica-neptune \
  --template-body file://neptune-setup.yaml \
  --parameters ParameterKey=ClientCidr,ParameterValue=203.0.113.25/32 \
  --capabilities CAPABILITY_NAMED_IAM

# 等待建栈完成
aws cloudformation wait stack-create-complete --stack-name semantica-neptune

# 取输出（endpoint、端口、凭据）
aws cloudformation describe-stacks --stack-name semantica-neptune \
  --query 'Stacks[0].Outputs' --output table
```

模板会创建：
- 带公网子网、Internet 网关和 VPC Flow Logs（发 CloudWatch Logs）的 VPC
- Neptune 集群（`db.t3.medium`），开启 IAM 认证
- 以最小权限做 OpenCypher 查询的 IAM 用户
- 只允许你指定的 `ClientCidr` 访问 Bolt 协议（8182 端口）的安全组

> ⚠️ **安全提示**：为简化演示/测试环境，此模板创建了带静态访问密钥的 IAM 用户。生产环境请用 IAM 角色（EC2 实例角色、ECS 任务角色、Lambda 执行角色）——临时凭据自动轮换。CloudFormation 输出里的 Secret Access Key 为明文，只是为了简化初次搭建——生产请用 AWS Secrets Manager。`ClientCidr` 参数必填（无默认值），正是为了让数据库绝不默默暴露给整个互联网。

**输出：**
- `NeptuneEndpoint` - 集群主机名（用作 `NEPTUNE_ENDPOINT`）
- `NeptunePort` - 8182（用作 `NEPTUNE_PORT`）
- `AwsAccessKeyId` - IAM 用户访问密钥（用作 `AWS_ACCESS_KEY_ID`）
- `AwsSecretAccessKey` - IAM 用户私有密钥，**明文**（用作 `AWS_SECRET_ACCESS_KEY`）
- `AwsRegion` - 部署区域（用作 `AWS_REGION`）

**清理：**
```bash
aws cloudformation delete-stack --stack-name semantica-neptune
```

**预估月成本（100% 使用率下约 100-105 美元/月）：**

| 资源 | 成本（美元） |
| --- | --- |
| Neptune db.t3.medium 实例 | ~96/月（0.132/时） |
| 存储（10 GB） | ~1/月 |
| I/O 请求 | ~1-5/月 |
| 公网 IPv4 地址 | ~3.60/月（0.005/时） |
| VPC Flow Logs（CloudWatch Logs） | ~1-2/月，随流量 |
| VPC、子网、路由表、Internet 网关、IAM | 不额外收费 |

> **免费额度**：Neptune 新用户有 30 天免费（750 小时 db.t3.medium、1000 万 I/O、1 GB 存储）。不用时删掉栈，避免扣费。

---

## 安装

```bash
# 安装带 Neptune 支持的 Semantica
pip install semantica

# 必需依赖（自动安装）
pip install boto3 neo4j
```

In [ ]:
!pip install semantica

## 配置

设置 Neptune 集群端点和 AWS 凭据。把占位值换成你的实际配置。

In [ ]:
import os

# Neptune cluster configuration - REPLACE WITH YOUR VALUES
# (Get these from CloudFormation stack outputs)
os.environ["NEPTUNE_ENDPOINT"] = "your-cluster.us-east-1.neptune.amazonaws.com"
os.environ["NEPTUNE_PORT"] = "8182"
os.environ["AWS_REGION"] = "us-east-1"

# AWS credentials for IAM Authentication
# Option 1: IAM User (static credentials from CloudFormation template)
# os.environ["AWS_ACCESS_KEY_ID"] = "AKIA..."  # From AwsAccessKeyId output
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."  # From AwsSecretAccessKey output
# Note: No AWS_SESSION_TOKEN needed for IAM users

# Option 2: IAM Role / Temporary credentials (e.g., STS AssumeRole, EC2 instance role)
# os.environ["AWS_ACCESS_KEY_ID"] = "ASIA..."  # Temporary access key
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."  # Temporary secret key
# os.environ["AWS_SESSION_TOKEN"] = "..."      # REQUIRED for temporary credentials

print(f"Neptune Endpoint: {os.environ.get('NEPTUNE_ENDPOINT')}")
print(f"AWS Region: {os.environ.get('AWS_REGION')}")

## 第 1 步：初始化 Neptune 存储

用 IAM 认证初始化到 Amazon Neptune 集群的连接。

In [ ]:
import os
from semantica.graph_store import GraphStore

# Option 1: Using GraphStore factory (recommended)
neptune_store = GraphStore(
    backend="neptune",
    endpoint=os.environ.get("NEPTUNE_ENDPOINT"),
    port=int(os.environ.get("NEPTUNE_PORT", 8182)),
    region=os.environ.get("AWS_REGION", "us-east-1"),
    iam_auth=True,
)

# Connect to Neptune
neptune_store.connect()
print("Connected to Amazon Neptune!")

### 开发/测试环境不带 IAM 认证

开发或测试环境如果不需要 IAM 认证（比如 Neptune notebook 或仅 VPC 内访问），可以关掉 IAM 签名：

In [ ]:
# For dev/test environments without IAM authentication
neptune_store_dev = GraphStore(
    backend="neptune",
    endpoint=os.environ.get("NEPTUNE_ENDPOINT"),
    port=int(os.environ.get("NEPTUNE_PORT", 8182)),
    region=os.environ.get("AWS_REGION", "us-east-1"),
    iam_auth=False,  # Disable IAM signing for dev/test
)
neptune_store_dev.connect()

### 认证选项

IAM 认证（生产推荐）自动使用 AWS 凭据链：
1. 环境变量（AWS_ACCESS_KEY_ID、AWS_SECRET_ACCESS_KEY）
2. AWS 凭据文件（~/.aws/credentials）
3. IAM 角色（EC2、Lambda、ECS）

## 第 2 步：节点操作

### 创建节点

节点代表图里的实体。每个节点可以有：
- **ID**：唯一标识（自定义或自动生成的 UUID）
- **标签（Labels）**：类别/类型（如 `Person`、`Company`）
- **属性（Properties）**：键值对（如 `{"name": "Alice", "age": 30}`）

Neptune 的规矩：自定义 ID 要把 `id` 放进 properties 里；不放就自动生成 UUID。

In [ ]:
# Create a single node with custom ID (id in properties)
alice = neptune_store.create_node(
    labels=["Person"],
    properties={"id": "alice", "name": "Alice", "age": 30, "role": "Engineer"}
)
print(f"Created node: {alice}")

# Create a node with auto-generated UUID (no id in properties)
bob = neptune_store.create_node(
    labels=["Person"],
    properties={"name": "Bob", "age": 25, "role": "Designer"}
)
print(f"Created node with UUID: {bob['id']}")

# Create a company node with auto-generated ID
acme = neptune_store.create_node(
    labels=["Company"],
    properties={"name": "Acme Corp", "industry": "Technology", "founded": 2010}
)
print(f"Created company: {acme}")

### 批量创建多个节点

In [ ]:
# Batch create nodes for better performance
# Include 'id' in properties for custom IDs
nodes_data = [
    {"labels": ["Person"], "properties": {"id": "charlie", "name": "Charlie", "age": 35}},
    {"labels": ["Person"], "properties": {"id": "diana", "name": "Diana", "age": 28}},
    {"labels": ["Location"], "properties": {"name": "San Francisco", "state": "CA"}},
]

created_nodes = neptune_store.create_nodes(nodes_data)
print(f"Created {len(created_nodes)} nodes in batch")

### 读取节点

In [ ]:
# Get a specific node by ID
alice_node = neptune_store.get_node(node_id="alice")
print(f"Retrieved: {alice_node}")

# Get nodes by label
people = neptune_store.get_nodes(labels=["Person"], limit=10)
print(f"Found {len(people)} Person nodes:")
for person in people:
    print(f"  - {person.get('properties', {}).get('name')}")

# Get nodes by properties
engineers = neptune_store.get_nodes(
    labels=["Person"],
    properties={"role": "Engineer"},
    limit=5
)
print(f"Found {len(engineers)} engineers")

### 更新节点

In [ ]:
# Update node properties (merge mode - default)
updated_alice = neptune_store.update_node(
    node_id="alice",
    properties={"age": 31, "department": "AI Research"},
    merge=True
)
print(f"Updated Alice: {updated_alice}")

# Replace all properties (merge=False)
# WARNING: This removes properties not in the update
replaced = neptune_store.update_node(
    node_id="charlie",
    properties={"name": "Charlie", "age": 36},
    merge=False
)

### 删除节点

In [ ]:
# Delete a node (with detach=True to also delete relationships)
deleted = neptune_store.delete_node(node_id="diana", detach=True)
print(f"Deleted diana: {deleted}")

# Without detach (fails if node has relationships)
# neptune_store.delete_node(node_id="alice", detach=False)

## 第 3 步：关系操作

### 创建关系

关系连接节点，代表实体之间的连接。

In [ ]:
# Create a relationship between Alice and Acme
works_at = neptune_store.create_relationship(
    start_node_id="alice",
    end_node_id=acme["id"],
    rel_type="WORKS_AT",
    properties={"since": 2020, "position": "Senior Engineer"}
)
print(f"Created relationship: {works_at}")

# Create a KNOWS relationship between people
knows_rel = neptune_store.create_relationship(
    start_node_id="alice",
    end_node_id=bob["id"],
    rel_type="KNOWS",
    properties={"since": 2019}
)

### 读取关系

In [ ]:
# Get all relationships for a node
alice_rels = neptune_store.get_relationships(node_id="alice", direction="both")
print(f"Alice has {len(alice_rels)} relationships")

# Get outgoing relationships only
outgoing = neptune_store.get_relationships(node_id="alice", direction="out")

# Filter by relationship type
works_rels = neptune_store.get_relationships(
    node_id="alice",
    rel_type="WORKS_AT",
    direction="out"
)
print(f"Alice's work relationships: {len(works_rels)}")

### 删除关系

In [ ]:
# Delete a specific relationship by ID
if works_at.get("id"):
    deleted = neptune_store.delete_relationship(rel_id=works_at["id"])
    print(f"Deleted relationship: {deleted}")

## 第 4 步：OpenCypher 查询

Amazon Neptune 经 Bolt 协议支持 OpenCypher。用标准 Cypher 语法表达复杂图模式——和 09 篇同一套写法。

In [ ]:
# Simple query
results = neptune_store.execute_query(
    "MATCH (p:Person) RETURN p.name, p.age ORDER BY p.age"
)
print("People in the graph:")
for record in results.get("records", []):
    print(f"  - {record.get('p.name')}: {record.get('p.age')} years old")

In [ ]:
# Using parameters (safer and more efficient)
results = neptune_store.execute_query(
    "MATCH (p:Person) WHERE p.age > $min_age RETURN p.name, p.age",
    parameters={"min_age": 25}
)
print(f"People over 25: {len(results.get('records', []))}")

In [ ]:
# Find relationships between nodes
results = neptune_store.execute_query("""
    MATCH (p:Person)-[r:WORKS_AT]->(c:Company)
    RETURN p.name as employee, c.name as company, r.since as start_year
""")
for record in results.get("records", []):
    print(f"{record['employee']} works at {record['company']} since {record['start_year']}")

In [ ]:
# Count and aggregate
results = neptune_store.execute_query("""
    MATCH (p:Person)
    RETURN count(p) as total, avg(p.age) as avg_age, max(p.age) as max_age
""")
stats = results.get("records", [{}])[0]
print(f"Total: {stats.get('total')}, Avg Age: {stats.get('avg_age'):.1f}")

## 第 5 步：图分析

### 获取邻居

遍历图找相连的节点。

In [ ]:
# Get immediate neighbors (depth=1)
neighbors = neptune_store.get_neighbors(
    node_id="alice",
    direction="both",
    depth=1
)
print(f"Alice's direct neighbors: {len(neighbors)}")

# Get neighbors up to 2 hops away
extended = neptune_store.get_neighbors(
    node_id="alice",
    direction="out",
    depth=2
)
print(f"Nodes within 2 hops: {len(extended)}")

### 最短路径

找两个节点之间的最短路。

In [ ]:
# Find shortest path
path = neptune_store.shortest_path(
    start_node_id="alice",
    end_node_id="charlie",
    max_depth=5
)

if path:
    print("Path found!")
    print(f"  Length: {path.get('length')}")
    print(f"  Nodes: {len(path.get('nodes', []))}")
    print(f"  Relationships: {len(path.get('relationships', []))}")
else:
    print("No path found between nodes")

## 第 6 步：图统计

获取图谱的全面统计——给图做体检，09 篇同款。

In [ ]:
# Get graph statistics
stats = neptune_store.get_stats()

print("Graph Statistics:")
print(f"  Total nodes: {stats.get('node_count', 'N/A')}")
print(f"  Total relationships: {stats.get('relationship_count', 'N/A')}")

print("\nNode labels:")
for label, count in stats.get('label_counts', {}).items():
    print(f"  - {label}: {count}")

print("\nRelationship types:")
for rel_type, count in stats.get('relationship_type_counts', {}).items():
    print(f"  - {rel_type}: {count}")

## 第 7 步：连接管理

用完记得关闭连接，释放资源。

In [ ]:
# Check connection status
status = neptune_store.get_status()
print(f"Connection status: {status}")

# Close the connection
neptune_store.close()
print("Connection closed")

## Neptune 特有事项

### 原生元素 ID

Neptune 用原生 `~id` 标识元素。在 properties 里带上 `id` 即可自定义 ID：

```python
# 自定义 ID（properties 里带 'id'）
node = neptune_store.create_node(
    labels=["Person"],
    properties={"id": "my-custom-id", "name": "Test"}
)

# 自动生成 UUID（properties 里不放 'id'）
node = neptune_store.create_node(
    labels=["Person"],
    properties={"name": "Test"}
)

# ID 会在内部的 id() 函数调用中使用：
# MATCH (n) WHERE id(n) = 'my-custom-id' RETURN n
```

### OpenCypher 注意事项

Amazon Neptune 数据库的 OpenCypher 实现与 Neo4j 有几处差异：

1. **没有 `shortestPath()` 函数**：用变长路径模式或 `allShortestPaths()`
2. **标签语法**：用 `labels(n)` 函数取节点标签
3. **属性更新**：用 `SET n += {props}` 实现 merge 行为

Neptune 数据库支持的完整 OpenCypher 规范见 [AWS 文档](https://docs.aws.amazon.com/neptune/latest/userguide/access-graph-opencypher.html)。

### Amazon Neptune Analytics

图算法、聚合、大规模遍历这类分析型（OLAP）负载，考虑 [Amazon Neptune Analytics](https://docs.aws.amazon.com/neptune-analytics/latest/userguide/what-is-neptune-analytics.html)。两者互补：Analytics 为分析查询优化性能，Database 为事务型（OLTP）负载优化。

### 性能技巧

1. 建多个节点/关系**用批量操作**
2. 查询里**用参数**，启用查询缓存
3. 用 `LIMIT` **限制结果集**

## 小结

### 回顾主线

云端孪生篇走完：**上云**（CloudFormation 开栈、IAM 认证——云上多出的入场券）、**同接口 CRUD**（节点、关系、OpenCypher、分析、统计——09 篇的节奏原样复用，多后端抽象的价值兑现）、**懂差异**（`~id`、OpenCypher 语法差异、OLTP/OLAP 分工）。还记得那三件云上小事吗——认证、网络白名单、成本，用完删栈。

本 notebook 讲了 Amazon Neptune 图存储集成：

- **IAM 认证**：安全的 AWS SigV4 签名
- **CRUD 操作**：完整的节点与关系管理
- **OpenCypher 查询**：标准图查询语言
- **图分析**：邻居与最短路径算法
- **统计与监控**：图指标与连接状态

### 核心要点

- Neptune 用原生 `~id` 标识元素
- 生产环境推荐 IAM 认证
- Bolt 协议提供高效的二进制查询接口
- Semantica 抹平了 Neptune 特有的语法差异

### 下一步

- [图存储（Neo4j/FalkorDB）](09_Graph_Store.ipynb) - 对比其他后端
- [构建知识图谱](07_Building_Knowledge_Graphs.ipynb) - 构建生产级 KG
- [图分析](10_Graph_Analytics.ipynb) - 高级分析算法